In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

In [3]:

data_path = Path(r"C:\Users\Alfonso(FonsoP)Panli\Desktop\Projects\Disengagement\Data")

# Read Data
sc_engagement_raw = pd.read_excel(Path(data_path) / 'SC Engagement List2026-10-07 07_48_09.xlsx')
mystaff_raw = pd.read_excel(Path(data_path) / 'CONFIDENTIAL Mystaff 10052026.xlsx', sheet_name='MyStaff')


In [76]:
mystaff_raw.head()

,Client,Shortcode,Timezone,Source,Account Manager/s,Current Business Development Manager,Virtual Account Manager,Date,Active,CRM Company ID,...,Industry Hubspot,Tenure,Current Segment 1,Current Segment 2,Current Segment 3,Current Segment 4,Peak Segment 1,Peak Segment 2,Peak Segment 3,Peak Segment 4
0,Customer Portal - QA Test v3,PHGENTERMS,NaN,Register,NaN,NaN,NaN,2026-10-02,No,5.888155e+10,...,-,1. 0-3 months,0. Boneyard or AirBridge,0. Boneyard or AirBridge,Inactive0,Inactive0,0. Boneyard or AirBridge,0. Boneyard or AirBridge,0. Boneyard or AirBridge,Inactive1
1,Settled Sales Pty Ltd,AUTHEPRESTIGEAG,NaN,Register,MimiA(James Adrienne Ancheta),PaulBur(Paul Burrows),Kim(Kim Perez),2026-10-02,No,5.888697e+10,...,-,1. 0-3 months,0. Boneyard or AirBridge,0. Boneyard or AirBridge,Inactive0,Inactive0,0. Boneyard or AirBridge,0. Boneyard or AirBridge,0. Boneyard or AirBridge,Inactive1
2,M&R Homes,PHCLOUDSTAFF2,NaN,Register,NaN,NaN,NaN,2026-10-02,No,5.882666e+10,...,-,1. 0-3 months,0. Boneyard or AirBridge,0. Boneyard or AirBridge,Inactive0,Inactive0,0. Boneyard or AirBridge,0. Boneyard or AirBridge,0. Boneyard or AirBridge,Inactive1
3,END TO END SYSTEM,19758247848,NaN,MyStaff,NaN,NaN,NaN,2026-10-01,No,NaN,...,-,1. 0-3 months,0. Boneyard or AirBridge,0. Boneyard or AirBridge,Inactive0,Inactive0,0. Boneyard or AirBridge,0. Boneyard or AirBridge,0. Boneyard or AirBridge,Inactive1
4,Gradius IT Solutions,USGRADIUSITSOLU,NaN,Register,NaN,NaN,NaN,2026-10-01,No,5.885303e+10,...,-,1. 0-3 months,0. Boneyard or AirBridge,0. Boneyard or AirBridge,Inactive0,Inactive0,0. Boneyard or AirBridge,0. Boneyard or AirBridge,0. Boneyard or AirBridge,Inactive1


In [80]:

today = pd.Timestamp.today().normalize()

# Data Formatting
sc_engagement = sc_engagement_raw.copy()
sc_engagement['engagement_date'] = pd.to_datetime(sc_engagement['Engagement Date'])
sc_engagement['disengagement_date'] = pd.to_datetime(sc_engagement['Disengagement Date'], errors="coerce").fillna(today)

mystaff_cols = ['Shortcode', 'Region', 'Industry Roll Up', 'Industry Hubspot']
mystaff = mystaff_raw[mystaff_cols].copy()

In [106]:
# ============================================================
# Setup
# ============================================================
exit_statuses = ["Resigned", "AWOL", "Dismissed"]   # "involuntary / unplanned" exits
today_ts = pd.Timestamp(today)

sc_engagement["engagement_date"] = pd.to_datetime(sc_engagement["engagement_date"])
sc_engagement["disengagement_date"] = pd.to_datetime(sc_engagement["disengagement_date"])

# ============================================================
# Build events
# ============================================================
# +1 when someone starts
starts = (sc_engagement[["Shortcode", "engagement_date"]]
          .rename(columns={"engagement_date": "date"})
          .copy())
starts["Employment Status"] = "Start Employment"
starts["delta"] = 1
starts["is_attrition"] = 0
starts["is_exit"] = 0                                   # NEW

# -1 when someone ends (skip rows that were filled with today = still active)
ended = sc_engagement[sc_engagement["disengagement_date"] != today_ts]
ends = (ended[["Shortcode", "disengagement_date", "Employment Status"]]
        .rename(columns={"disengagement_date": "date"})
        .copy())
ends["delta"] = -1
ends["is_attrition"] = ends["Employment Status"].isin(exit_statuses).astype(int)
ends["is_exit"] = 1                                     # NEW: any disengagement

events = pd.concat([starts, ends], ignore_index=True)

# ============================================================
# Headcount timeline
# ============================================================
timeline = (
    events.groupby(["Shortcode", "date"], as_index=False)
          .agg(delta=("delta", "sum"),
               is_attrition=("is_attrition", "max"),   # 1 if any attrition exit on that date
               is_exit=("is_exit", "max"))             # NEW: 1 if any exit on that date
          .sort_values(["Shortcode", "date"])
          .reset_index(drop=True)
)
timeline["active_staff"] = timeline.groupby("Shortcode")["delta"].cumsum()
timeline = timeline.drop(columns="delta")

# ============================================================
# Engagement summary
# ============================================================
# --- Core start / end info ---
summary = (
    timeline.groupby("Shortcode")
            .agg(start_date=("date", "first"),
                 employee_count_at_start=("active_staff", "first"),
                 last_date=("date", "last"),
                 current_headcount=("active_staff", "last"))
            .reset_index()
)

summary["start_cohort"] = summary["start_date"].dt.to_period("Q")
summary["engagement_active"] = summary["current_headcount"].gt(0).astype(int)
summary["end_date"] = summary["last_date"].where(summary["engagement_active"] == 0, pd.NaT)
summary["engagement_timespan_days"] = (
    (summary["end_date"] - summary["start_date"]).dt.days.astype("Int64")
)
summary["days_observed"] = (today_ts - summary["start_date"]).dt.days

# --- Peak headcount ---
peak_idx = timeline.groupby("Shortcode")["active_staff"].idxmax()
peak = (timeline.loc[peak_idx, ["Shortcode", "date", "active_staff"]]
                .rename(columns={"date": "peak_date", "active_staff": "peak_headcount"}))
summary = summary.merge(peak, on="Shortcode", how="left")
summary["days_to_peak"] = (summary["peak_date"] - summary["start_date"]).dt.days

# --- Headcount milestones ---
for n in (5, 10):
    first_hit = (timeline[timeline["active_staff"] >= n]
                 .groupby("Shortcode")["date"].min())
    summary[f"days_till_{n}_headcount"] = (
        (summary["Shortcode"].map(first_hit) - summary["start_date"])
        .dt.days.astype("Int64")
    )

# --- Attrition volume ---
hires = sc_engagement.groupby("Shortcode").size()
exits = ends.groupby("Shortcode").size()
attr_exits = ends.groupby("Shortcode")["is_attrition"].sum()

summary["total_hires"] = summary["Shortcode"].map(hires)
summary["total_exits"] = summary["Shortcode"].map(exits).fillna(0).astype(int)
summary["attrition_exits"] = summary["Shortcode"].map(attr_exits).fillna(0).astype(int)
summary["attrition_rate"] = (summary["attrition_exits"] / summary["total_hires"]).round(3)

by_type = (ends[ends["is_attrition"] == 1]
           .pivot_table(index="Shortcode", columns="Employment Status",
                        values="date", aggfunc="count", fill_value=0)
           .reindex(columns=exit_statuses, fill_value=0)   # all 3 columns always present
           .add_prefix("exits_")
           .reset_index())
summary = summary.merge(by_type, on="Shortcode", how="left")
type_cols = [f"exits_{s}" for s in exit_statuses]
summary[type_cols] = summary[type_cols].fillna(0).astype(int)

# --- Attrition timing ---
t = timeline.merge(summary[["Shortcode", "start_date"]], on="Shortcode")
t["days_since_start"] = (t["date"] - t["start_date"]).dt.days
attr = t[t["is_attrition"] > 0]

first_attr = attr.groupby("Shortcode")["days_since_start"].min()
summary["days_to_first_attrition"] = summary["Shortcode"].map(first_attr).astype("Int64")

for d in (30, 60, 90):
    flagged = attr.loc[attr["days_since_start"] <= d, "Shortcode"].unique()
    summary[f"eligible_{d}d"] = (summary["days_observed"] >= d).astype(int)
    summary[f"attrition_within_{d}d"] = summary["Shortcode"].isin(flagged).astype(int)

# --- Early disengagement timing (any exit)  NEW ---
exit_ev = t[t["is_exit"] > 0]

first_exit = exit_ev.groupby("Shortcode")["days_since_start"].min()
summary["days_to_first_disengagement"] = summary["Shortcode"].map(first_exit).astype("Int64")

for d in (30, 60, 90):
    flagged = exit_ev.loc[exit_ev["days_since_start"] <= d, "Shortcode"].unique()
    summary[f"disengaged_within_{d}d"] = summary["Shortcode"].isin(flagged).astype(int)

# --- Column order ---
col_order = [
    # Identity
    "Shortcode", "start_cohort",
    # Lifecycle
    "start_date", "end_date", "engagement_active",
    "engagement_timespan_days", "days_observed",
    # Headcount growth
    "employee_count_at_start", "current_headcount",
    "peak_headcount", "peak_date", "days_to_peak",
    "days_till_5_headcount", "days_till_10_headcount",
    # Attrition volume
    "total_hires", "total_exits", "attrition_exits", "attrition_rate",
    *type_cols,
    # Attrition / disengagement timing
    "days_to_first_attrition", "days_to_first_disengagement",
    "eligible_30d", "attrition_within_30d", "disengaged_within_30d",
    "eligible_60d", "attrition_within_60d", "disengaged_within_60d",
    "eligible_90d", "attrition_within_90d", "disengaged_within_90d",
]
summary = summary[col_order]

# ============================================================
# Merge region / industry and move next to Shortcode
# ============================================================
print(summary.shape)
summary = summary.merge(mystaff, how="left")
move = ["Region", "Industry Roll Up", "Industry Hubspot"]
rest = [c for c in summary.columns if c not in ["Shortcode"] + move]
summary = summary[["Shortcode"] + move + rest]
print(summary.shape)

# ============================================================
# Cohort helpers
# ============================================================
summary["start_year"] = summary["start_date"].dt.year
summary["start_size_bucket"] = pd.cut(
    summary["employee_count_at_start"],
    bins=[0, 2, 5, float("inf")], labels=["1-2", "3-5", "6+"]
)

# Fixed-window ramp flags (eligible-only: NaN if not enough time observed)
for n in (5, 10):
    for d in (90, 180):
        hit = (summary[f"days_till_{n}_headcount"] <= d).fillna(False).astype(int)
        summary[f"reached_{n}_in_{d}d"] = hit.where(summary["days_observed"] >= d)

# Mask attrition and disengagement flags the same way
for d in (30, 60, 90):
    summary[f"attr_{d}d_elig"] = summary[f"attrition_within_{d}d"].where(
        summary[f"eligible_{d}d"] == 1)
    summary[f"disengaged_{d}d_elig"] = summary[f"disengaged_within_{d}d"].where(   # NEW
        summary[f"eligible_{d}d"] == 1)

# ============================================================
# Sanity checks
# ============================================================
for d in (30, 60, 90):
    # attrition is a subset of disengagement, so this should be 0 rows
    bad = summary[summary[f"attrition_within_{d}d"] > summary[f"disengaged_within_{d}d"]]
    print(f"{d}d: attrition > disengagement rows = {len(bad)}")

    elig = summary[summary[f"eligible_{d}d"] == 1]
    print(f"{d}d: eligible={len(elig)}, "
          f"attrition events={elig[f'attrition_within_{d}d'].sum()}, "
          f"disengagement events={elig[f'disengaged_within_{d}d'].sum()}")

# ============================================================
# Optional: replaces the old `ramp` block
# ============================================================
n_staff_start = 3
ramp = summary[(summary["employee_count_at_start"] < n_staff_start)
               & summary["days_till_5_headcount"].notna()]

# ============================================================
# Cohort rollup (reusable for any grouping)
# ============================================================
def rollup(df, by, min_n=20):
    out = df.groupby(by, observed=True).agg(
        engagements=("Shortcode", "count"),
        pct_active=("engagement_active", "mean"),
        median_start_size=("employee_count_at_start", "median"),
        median_peak=("peak_headcount", "median"),
        reached_5_90d=("reached_5_in_90d", "mean"),
        reached_10_180d=("reached_10_in_180d", "mean"),
        attr_30d=("attr_30d_elig", "mean"),
        attr_60d=("attr_60d_elig", "mean"),
        attr_90d=("attr_90d_elig", "mean"),
        disengaged_30d=("disengaged_30d_elig", "mean"),   # NEW
        disengaged_60d=("disengaged_60d_elig", "mean"),   # NEW
        disengaged_90d=("disengaged_90d_elig", "mean"),   # NEW
        n_90d_eligible=("eligible_90d", "sum"),
        total_hires=("total_hires", "sum"),
        attrition_exits=("attrition_exits", "sum"),
        resigned=("exits_Resigned", "sum"),
        awol=("exits_AWOL", "sum"),
        dismissed=("exits_Dismissed", "sum"),
    )
    out["attrition_per_hire"] = (out["attrition_exits"] / out["total_hires"]).round(3)
    return out[out["engagements"] >= min_n].round(3)

by_year     = rollup(summary, "start_year")
by_region   = rollup(summary, ["Region", "start_year"])
by_industry = rollup(summary, ["Industry Roll Up", "start_year"])
by_size     = rollup(summary, "start_size_bucket")
by_reg_ind  = rollup(summary, ["Region", "Industry Roll Up"], min_n=30)

(2481, 32)
(2481, 35)
30d: attrition > disengagement rows = 0
30d: eligible=2455, attrition events=97, disengagement events=199
60d: attrition > disengagement rows = 0
60d: eligible=2426, attrition events=191, disengagement events=358
90d: attrition > disengagement rows = 0
90d: eligible=2399, attrition events=290, disengagement events=520


In [108]:
summary.to_clipboard()

In [97]:
def rollup(df, by, min_n=20):
    out = df.groupby(by, observed=True).agg(
        engagements=("Shortcode", "count"),
        pct_active=("engagement_active", "mean"),
        median_start_size=("employee_count_at_start", "median"),
        median_peak=("peak_headcount", "median"),
        reached_5_90d=("reached_5_in_90d", "mean"),
        reached_10_180d=("reached_10_in_180d", "mean"),
        attr_30d=("attr_30d_elig", "mean"),
        attr_60d=("attr_60d_elig", "mean"),
        attr_90d=("attr_90d_elig", "mean"),
        n_90d_eligible=("eligible_90d", "sum"),
        total_hires=("total_hires", "sum"),
        attrition_exits=("attrition_exits", "sum"),
        resigned=("exits_Resigned", "sum"),
        awol=("exits_AWOL", "sum"),
        dismissed=("exits_Dismissed", "sum"),
    )
    out["attrition_per_hire"] = (out["attrition_exits"] / out["total_hires"]).round(3)
    return out[out["engagements"] >= min_n].round(3)

by_year      = rollup(summary, "start_year")
by_region    = rollup(summary, ["Region", "start_year"])
by_industry  = rollup(summary, ["Industry Roll Up", "start_year"])
by_size      = rollup(summary, "start_size_bucket")
by_reg_ind   = rollup(summary, ["Region", "Industry Roll Up"], min_n=30)

In [102]:
# Predictor window: first 30 days. Outcome window: by day 180.
lm = summary[
    (summary["days_observed"] >= 180) &                   # fully observable
    ~(summary["days_till_5_headcount"] <= 30)             # hadn't already hit 5
].copy()

lm["reached_5_by_180"] = (lm["days_till_5_headcount"] <= 180).fillna(False).astype(int)

# Raw comparison
lm.groupby("attrition_within_30d")["reached_5_by_180"].agg(n="count", rate="mean")

# Controlling for the biggest confounder: starting size
(lm.groupby(["start_size_bucket", "attrition_within_30d"], observed=True)
   ["reached_5_by_180"].agg(n="count", rate="mean"))

n      rate
start_size_bucket attrition_within_30d               
1-2               0                     434  0.258065
                  1                       4  0.500000
3-5               0                      28  0.500000

In [103]:
from scipy.stats import fisher_exact

sub = lm[lm["start_size_bucket"] == "1-2"]
tab = pd.crosstab(sub["attrition_within_30d"], sub["reached_5_by_180"])
print(tab)
print(fisher_exact(tab))   # (odds_ratio, p_value)

reached_5_by_180        0    1
attrition_within_30d          
0                     322  112
1                       2    2
SignificanceResult(statistic=np.float64(2.875), pvalue=np.float64(0.2787378712302765))


In [ ]:
# Possible next steps
# - identify if an employee resigns with 0-30, 0-60 of theyre employment


In [51]:
summary.to_clipboard()

In [39]:
summary[summary['attrition_within_30d']>0]

,Shortcode,start_date,employee_count_at_start,engagement_ended,end_date,engagement_timespan_days,days_till_5_headcount,days_till_10_headcount,attrition_within_30d,attrition_within_60d,attrition_within_90d
48,AGENTSAGENCY,2024-01-31,2,0,NaT,<NA>,124,420,1,1,1
55,ALEXILUM,2024-03-11,1,0,NaT,<NA>,<NA>,<NA>,1,1,1
157,AUSPHIL,2019-07-16,31,1,2021-09-12,789,0,0,1,1,1
178,AVONTUS,2021-04-05,1,1,2021-04-16,11,<NA>,<NA>,1,1,1
187,BAKERMILLIGAN,2022-10-17,1,1,2022-10-21,4,<NA>,<NA>,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...
2416,WASHINGTONWEST,2026-01-19,1,0,NaT,<NA>,<NA>,<NA>,1,1,1
2421,WEALTHYYOU,2020-02-17,1,1,2020-03-02,14,<NA>,<NA>,1,1,1
2440,WISEINST,2025-09-15,1,1,2025-10-02,17,<NA>,<NA>,1,1,1
2443,WOJACK,2022-05-02,1,1,2022-05-21,19,<NA>,<NA>,1,1,1


In [26]:
timeline[timeline['is_attrition']==1]

,Shortcode,date,is_attrition,active_staff
6,05655543,2016-03-14,1,9
9,05655543,2016-03-30,1,10
16,05655543,2016-06-16,1,15
18,05655543,2016-06-30,1,15
19,05655543,2016-07-05,1,14
...,...,...,...,...
29758,ZUMI,2019-11-25,1,9
29760,ZUMI,2020-01-19,1,7
29761,ZUMI,2020-01-31,1,5
29763,ZUMI,2020-02-18,1,3


In [13]:
sc_engagement['Employment Status'].unique()

<StringArray>
[         'Contractual',            'Separated',             'Resigned',
         'Probationary',            'Part-Time',       'End-of-Project',
              'Regular',      'End-of-Contract',                 'AWOL',
                  'TLO', 'Cancelled Employment',     'End of Probation',
            'Dismissed']
Length: 13, dtype: str

In [6]:
ended['Employment Status'].unique()

<StringArray>
[           'Separated',             'Resigned',       'End-of-Project',
      'End-of-Contract',                 'AWOL',              'Regular',
                  'TLO',         'Probationary', 'Cancelled Employment',
          'Contractual',            'Dismissed',     'End of Probation',
            'Part-Time']
Length: 13, dtype: str

In [ ]:
# Next steps
# What is quick vs slow, maybe tiers of days
# - pull in features from Mystaff
#   - which of these features are good indicators of ramping from 1 to 5 quickly
#   - Employee disengagement??? (contract void, employee resigns within n,m,p days??)

In [101]:
ramp["days_to_5"].describe()
ramp["days_to_5"].median()
ramp["completed"].value_counts()

completed
True    480
Name: count, dtype: int64

In [103]:
ramp["days_to_5"].describe()


count         480.0
mean     607.777083
std      702.421988
min             1.0
25%          114.25
50%           351.0
75%           887.5
max          6694.0
Name: days_to_5, dtype: Float64

In [47]:
under_n.sort_values('active_staff')

,Shortcode,date,active_staff
474,CIB187,2018-03-20,-1
33,ADESSOCAPITAL,2025-09-22,0
2384,VERIFYDEBTSOLN,2023-11-20,0
2399,VIMWOOD,2024-11-11,0
2115,SEETORE,2023-10-31,0
...,...,...,...
1731,OPILINDUMPKNG,2024-11-11,2
2164,SNAPPY,2024-08-19,2
1878,POPUPTALENT,2021-02-16,2
2463,YOURINSURANCE,2020-02-24,2


In [66]:
timeline

,Shortcode,date,active_staff
0,05655543,2014-06-02,1
1,05655543,2014-08-14,2
2,05655543,2015-11-19,7
3,05655543,2016-01-04,8
4,05655543,2016-01-06,9
...,...,...,...
29765,ingenUIT,2024-02-26,1
29766,ingenUIT,2024-04-03,0
29767,opatdgenvi,2026-03-09,1
29768,opatdgenvi,2026-09-17,2


In [ ]:
# Data check

In [90]:
shortcode = 'CIB106'

In [94]:
events[events['Shortcode']==shortcode].sort_values(by='date').head(18)

,Shortcode,date,Employment Status,delta,is_attrition
17462,CIB106,2016-04-29,Start Employment,1,0
29896,CIB106,2016-05-06,Start Employment,1,0
23799,CIB106,2016-05-06,Start Employment,1,0
19195,CIB106,2016-05-06,Start Employment,1,0
15173,CIB106,2016-05-06,Start Employment,1,0
29061,CIB106,2016-05-06,Start Employment,1,0
22167,CIB106,2016-05-06,Start Employment,1,0
22095,CIB106,2016-05-06,Start Employment,1,0
8533,CIB106,2016-05-06,Start Employment,1,0
8261,CIB106,2016-05-06,Start Employment,1,0


In [92]:
timeline[timeline['Shortcode']==shortcode].sort_values(by='date').head(14)

,Shortcode,date,is_attrition,active_staff
3993,CIB106,2016-04-29,0,1
3994,CIB106,2016-05-06,0,14
3995,CIB106,2016-05-09,0,17
3996,CIB106,2016-05-12,0,20
3997,CIB106,2016-05-25,0,21
3998,CIB106,2016-06-18,1,18
3999,CIB106,2016-06-22,0,22
4000,CIB106,2016-06-26,1,21
4001,CIB106,2016-06-27,1,20
4002,CIB106,2016-07-03,1,19


In [93]:
sc_engagement[sc_engagement['Shortcode']==shortcode].sort_values(by='engagement_date').head(14)

,Username,CSID,User Status,First Name,Middle Name,Last Name,Employment Status,Category,Engagement ID,Client Name,Shortcode,Engagement Status,Engagement Date,Disengagement Date,engagement_date,disengagement_date
17462,kevinm@cloudstaff.com,CS-00984,inactive,kevin phillip,dayrit,manansala,Resigned,RTG,CS-00984-01,Turnkey Auto Group,CIB106,disengage,2016-04-29,2016-07-14,2016-04-29,2016-07-14
23799,OscarM@cloudstaff.com,CS-01001,active,oscar,bulaon,miranda,Regular,Engaged,CS-01001-01,Turnkey Auto Group,CIB106,disengage,2016-05-06,2016-07-14,2016-05-06,2016-07-14
22167,mikeec@cloudstaff.com,CS-00992,inactive,mikee james,gomez,corpin,Resigned,RTG,CS-00992-01,Turnkey Auto Group,CIB106,disengage,2016-05-06,2016-07-14,2016-05-06,2016-07-14
22095,MiguelM@cloudstaff.com,CS-01164,active,miguel,sicat,manarang,Regular,Engaged,CS-01164-01,Turnkey Auto Group,CIB106,disengage,2016-05-06,2016-07-14,2016-05-06,2016-07-14
19195,LisherM@cloudstaff.com,CS-00999,active,lisher mae,del pilar,martinez,Regular,Hub,CS-00999-01,Turnkey Auto Group,CIB106,disengage,2016-05-06,2016-07-14,2016-05-06,2016-07-14
29061,theresab@cloudstaff.com,CS-00991,inactive,maria theresa,samia,bognot,Resigned,RTG,CS-00991-01,Turnkey Auto Group,CIB106,disengage,2016-05-06,2016-07-14,2016-05-06,2016-07-14
15173,JomarB@cloudstaff.com,CS-00990,passive,jomar,baun,balagtas,TLO,Engaged,CS-00990-01,Turnkey Auto Group,CIB106,disengage,2016-05-06,2016-07-14,2016-05-06,2016-07-14
29896,vinap@cloudstaff.com,CS-01003,inactive,vina,manuel,pelayo,Resigned,Engaged,CS-01003-01,Turnkey Auto Group,CIB106,disengage,2016-05-06,2016-06-27,2016-05-06,2016-06-27
7796,edwardm@cloudstaff.com,CS-00997,inactive,edward lyle,velasco,mallare,Resigned,RTG,CS-00997-01,Turnkey Auto Group,CIB106,disengage,2016-05-06,2016-07-14,2016-05-06,2016-07-14
5477,christinea@cloudstaff.com,CS-00989,inactive,christine,caranay,adriano,Resigned,RTG,CS-00989-01,Turnkey Auto Group,CIB106,disengage,2016-05-06,2016-07-14,2016-05-06,2016-07-14
